In [1]:
import pandas as pd
import numpy as np
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import joblib

print("Constructing Master Dataset and Training XGBoost...")

# 1. Load the Base Data and Engineered Features
matches = pd.read_csv('../data/raw/matches.csv')
venue_intel = pd.read_csv('../data/processed/venue_intelligence.csv')
form_df = pd.read_csv('../data/processed/team_form.csv')
dominance_df = pd.read_csv('../data/processed/dominance_matrix.csv')

# Clean Team Names in base matches
team_mapping = {
    'Delhi Daredevils': 'Delhi Capitals', 'Kings XI Punjab': 'Punjab Kings', 
    'Deccan Chargers': 'Sunrisers Hyderabad', 'Rising Pune Supergiants': 'Rising Pune Supergiant',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru', 'Gujarat Lions': 'Gujarat Titans'
}
matches.replace(team_mapping, inplace=True)
matches.dropna(subset=['winner', 'venue'], inplace=True)

# 2. Build the Target Variable (Did Team 1 Win?)
matches['target'] = (matches['team1'] == matches['winner']).astype(int)

# 3. Merge Features (Simplified for the training pipeline)
# To keep this robust, we will map the venue win % directly into the matches dataframe
venue_win_dict = dict(zip(venue_intel['venue'], venue_intel['bat_first_win_pct']))
matches['venue_bat_first_win_pct'] = matches['venue'].map(venue_win_dict).fillna(50.0)

# 4. Select Final Features for the Pre-Match Engine
# We use the raw text columns (Pipeline handles encoding) + our engineered numerical columns
ml_df = matches[['team1', 'team2', 'venue', 'toss_winner', 'toss_decision', 'venue_bat_first_win_pct', 'target']].copy()

# Add a basic toss winner flag (1 if Team 1 won toss, 0 if Team 2)
ml_df['team1_toss_win'] = (ml_df['toss_winner'] == ml_df['team1']).astype(int)
ml_df.drop('toss_winner', axis=1, inplace=True)

# 5. Train-Test Split
X = ml_df.drop('target', axis=1)
y = ml_df['target']

# We use chronologically splitting instead of random so we don't predict the past using the future!
split_idx = int(len(ml_df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# 6. Create the Pipeline
step1 = ColumnTransformer([
    ('trf', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'), 
     ['team1', 'team2', 'venue', 'toss_decision'])
], remainder='passthrough')

# XGBoost Classifier
step2 = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

pipe_xgb = Pipeline([
    ('step1', step1),
    ('step2', step2)
])

# 7. Train and Evaluate
print("Training XGBoost Pipeline...")
pipe_xgb.fit(X_train, y_train)

y_pred = pipe_xgb.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"🔥 XGBoost Pre-Match Accuracy: {accuracy * 100:.2f}%")

# 8. Save the Model
os.makedirs('../models', exist_ok=True)
joblib.dump(pipe_xgb, '../models/xgboost_pre_match.pkl')
joblib.dump(list(X.columns), '../models/xgb_pre_match_features.pkl')

print("✅ Model saved to 'models/xgboost_pre_match.pkl'")

Constructing Master Dataset and Training XGBoost...
Training XGBoost Pipeline...


c:\Users\garvi\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:38:42] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


🔥 XGBoost Pre-Match Accuracy: 47.71%
✅ Model saved to 'models/xgboost_pre_match.pkl'


c:\Users\garvi\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
